# 🏏 PSL ML Project
## Notebook 5: Prediction Pipeline
Yahan hum ek easy-to-use function banein ge jo naye match data deke prediction le sake.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# Saved models load karo
rf_model = joblib.load('E:/PSL_Predictor/models/random_forest_model.pkl')
scaler = joblib.load('E:\PSL_Predictor\models\scaler.pkl')
with open('E:\PSL_Predictor\models\ feature_list.json') as f:
    FEATURES = json.load(f)

# Training data load karo (encoding ke liye)
df = pd.read_csv('E:\PSL_Predictor\data\psl_match_features.csv')
print('✅ Models aur data load ho gaye!')

OSError: [Errno 22] Invalid argument: 'E:\\PSL_Predictor\\models\x0ceature_list.json'

In [ ]:
# Team aur venue mappings
TEAM_MAP = {
    'Islamabad United': 0,
    'Karachi Kings': 1,
    'Lahore Qalandars': 2,
    'Multan Sultans': 3,
    'Peshawar Zalmi': 4,
    'Quetta Gladiators': 5
}
print('Available Teams:')
for team in sorted(TEAM_MAP.keys()):
    print(f'  - {team}')

In [ ]:
def predict_match_winner(
    batting_team,       # 1st innings batting team
    bowling_team,       # 1st innings bowling team
    venue_enc,          # venue encoding (0-9)
    season,             # PSL season year
    inn1_total_runs,    # 1st innings total runs
    inn1_wickets,       # 1st innings wickets fallen
    inn1_extras,        # 1st innings extras
    inn1_sixes,         # sixes in 1st innings
    inn1_fours,         # fours in 1st innings
    inn1_dot_balls,     # dot balls in 1st innings
    inn1_pp_runs,       # powerplay (1-6 overs) runs
    inn1_pp_wickets,    # powerplay wickets
    inn1_death_runs,    # death overs (16-20) runs
    inn1_death_wickets  # death overs wickets
):
    """
    PSL match winner predict karo 1st innings ke baad.
    Returns: (predicted_winner, win_probability)
    """
    # Derived features
    inn1_run_rate = inn1_total_runs / 20
    inn1_boundary_pct = (inn1_fours + inn1_sixes) / 120 * 100
    inn1_dot_ball_pct = inn1_dot_balls / 120 * 100
    pp_run_rate = inn1_pp_runs / 6
    death_run_rate = inn1_death_runs / 5
    
    # Input row banana
    input_data = pd.DataFrame([{
        'batting_team_enc': TEAM_MAP.get(batting_team, 0),
        'bowling_team_enc': TEAM_MAP.get(bowling_team, 0),
        'venue_enc': venue_enc,
        'season': season,
        'inn1_total_runs': inn1_total_runs,
        'inn1_wickets': inn1_wickets,
        'inn1_extras': inn1_extras,
        'inn1_sixes': inn1_sixes,
        'inn1_fours': inn1_fours,
        'inn1_dot_balls': inn1_dot_balls,
        'inn1_pp_runs': inn1_pp_runs,
        'inn1_pp_wickets': inn1_pp_wickets,
        'inn1_death_runs': inn1_death_runs,
        'inn1_death_wickets': inn1_death_wickets,
        'inn1_run_rate': inn1_run_rate,
        'inn1_boundary_pct': inn1_boundary_pct,
        'inn1_dot_ball_pct': inn1_dot_ball_pct,
        'pp_run_rate': pp_run_rate,
        'death_run_rate': death_run_rate
    }])[FEATURES]
    
    # Prediction
    pred = rf_model.predict(input_data)[0]
    prob = rf_model.predict_proba(input_data)[0]
    
    winner = batting_team if pred == 1 else bowling_team
    win_prob = prob[1] if pred == 1 else prob[0]
    
    return winner, win_prob

print('✅ Prediction function ready!')

In [ ]:
# ─── EXAMPLE PREDICTION ───────────────────────────────────────
# Islamabad United ne 175 runs banaye vs Karachi Kings

winner, prob = predict_match_winner(
    batting_team='Islamabad United',
    bowling_team='Karachi Kings',
    venue_enc=2,
    season=2024,
    inn1_total_runs=175,
    inn1_wickets=4,
    inn1_extras=8,
    inn1_sixes=10,
    inn1_fours=15,
    inn1_dot_balls=35,
    inn1_pp_runs=52,
    inn1_pp_wickets=1,
    inn1_death_runs=55,
    inn1_death_wickets=2
)

print('=' * 50)
print('🏏 PSL MATCH PREDICTION')
print('=' * 50)
print(f'1st Innings: Islamabad United scored 175 runs')
print(f'\n🏆 Predicted Winner: {winner}')
print(f'📊 Win Probability: {prob*100:.1f}%')
print('=' * 50)

In [ ]:
# Karachi Kings ne 140 runs banaye vs Lahore Qalandars

winner2, prob2 = predict_match_winner(
    batting_team='Karachi Kings',
    bowling_team='Lahore Qalandars',
    venue_enc=1,
    season=2024,
    inn1_total_runs=140,
    inn1_wickets=8,
    inn1_extras=5,
    inn1_sixes=5,
    inn1_fours=12,
    inn1_dot_balls=55,
    inn1_pp_runs=38,
    inn1_pp_wickets=3,
    inn1_death_runs=32,
    inn1_death_wickets=4
)

print('=' * 50)
print('🏏 PSL MATCH PREDICTION')
print('=' * 50)
print(f'1st Innings: Karachi Kings scored 140 runs')
print(f'\n🏆 Predicted Winner: {winner2}')
print(f'📊 Win Probability: {prob2*100:.1f}%')
print('=' * 50)

## 🎉 Project Complete!

### Project Summary:
| Notebook | Description |
|----------|-------------|
| `01_data_loading.ipynb` | Dataset load, columns, missing values |
| `02_eda.ipynb` | Visualizations — teams, players, overs, seasons |
| `03_preprocessing.ipynb` | Feature engineering, encoding, correlation |
| `04_model_training.ipynb` | 5 models train, compare, evaluate |
| `05_prediction.ipynb` | Ready-to-use prediction pipeline |

### Models Used:
- Logistic Regression
- Decision Tree
- Random Forest ✅ (Best)
- Gradient Boosting
- SVM

### Possible Improvements:
- XGBoost / LightGBM add karo
- Player-level features add karo
- Head-to-head team stats
- Hyperparameter tuning (GridSearchCV)
- SHAP values for explainability